# Guided Lab: ACO for TSP — TSPLIB `berlin52`
**Course:** SD234503 · **Estimated time:** 90–120 minutes

**Goal:** implement the Ant System variant of ACO and compare it with nearest-neighbor and the known optimum (7542).

## 1. Setup and dataset
Run cells in order. The notebook retrieves the official TSPLIB `.tsp` instance on first run. TSPLIB `EUC_2D` weights use integer rounding, so use `problem.get_weight()` rather than raw Euclidean floating-point distances.

In [ ]:
%pip -q install numpy pandas matplotlib tsplib95 requests

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import tsplib95
import requests
from pathlib import Path

# 1. Load TSPLIB Dataset

DATASET_URLS = {
    "berlin52.tsp":
        "https://raw.githubusercontent.com/ProvideQ/tsp-problems/master/berlin52.tsp",
    "eil76.tsp":
        "https://raw.githubusercontent.com/mastqe/tsplib/master/eil76.tsp"
}

def load_tsplib(filename):
    # Create dataset directory
    data_dir = Path("datasets")
    data_dir.mkdir(parents=True, exist_ok=True)

    path = data_dir / filename

    # Download only if dataset does not exist
    if not path.exists():
        if filename not in DATASET_URLS:
            raise ValueError(
                f"Unknown dataset: {filename}. "
                "Please provide a valid TSPLIB dataset."
            )
        print(f"Downloading {filename}...")
        url = DATASET_URLS[filename]
        response = requests.get(url, timeout=30)
        response.raise_for_status()

        # Check TSPLIB file format
        content = response.text
        if (
            "NODE_COORD_SECTION" not in content
            and "EDGE_WEIGHT_SECTION" not in content
        ):
            raise ValueError(
                "Downloaded file is not a valid TSPLIB dataset."
            )
        path.write_bytes(response.content)
        print(f"Dataset saved to: {path}")
    else:
        print(f"Loading existing dataset: {path}")

    # Read TSPLIB dataset
    problem = tsplib95.load(str(path))
    nodes = list(problem.get_nodes())

    # Official TSPLIB distance matrix
    D = np.array([
        [problem.get_weight(i, j) for j in nodes]
        for i in nodes
    ], dtype=float)

    # City coordinates
    coords = (
        np.array(
            [problem.node_coords[i] for i in nodes],
            dtype=float
        )
        if problem.node_coords
        else None
    )

    return problem, nodes, D, coords

# 2. Calculate Tour Length


# 3. Validate TSP Tour

# 4. Visualize TSP Tour

# 5. Nearest Neighbor Baseline


In [ ]:
# Step 1: Load the Berlin52 dataset
# D contains distances between cities, and coords contains city locations.

# Step 2: Display basic dataset information

# Step 3: Check that the dataset is valid

# Step 4: Visualize the locations of all cities


## 2. TSP objective
For a tour $\pi$, minimize $L(\pi)=\sum_{k=1}^{n-1}d_{\pi_k,\pi_{k+1}}+d_{\pi_n,\pi_1}$. The final term returns to the starting city. A valid tour visits every city exactly once.

In [ ]:
# Step 1: Create a sample route visiting cities in their original order
# Step 2: Check that every city is visited exactly once
# Step 3: Calculate the total route distance
# Includes returning from the last city to the starting city

# TODO: Write your Python code below.


## 3. Nearest-neighbor baseline
At each step, travel to the closest unvisited city. Repeat from every start to obtain a stronger baseline; it is still a heuristic.

In [ ]:
# Step 1: Run Nearest-Neighbor from every possible starting city
# Store the total distance and starting city for each route

# Step 2: Find the shortest route among all starting cities

# Step 3: Generate the best Nearest-Neighbor route

# Step 4: Display the best route distance and starting city

# Step 5: Visualize the best Nearest-Neighbor route


## 4. Ant Colony Optimization: mathematical mechanism
For feasible next city $j$, the transition probability is
$$p_{ij}=\frac{\tau_{ij}^{\alpha}\eta_{ij}^{\beta}}{\sum_{k\in A_i}\tau_{ik}^{\alpha}\eta_{ik}^{\beta}},\qquad\eta_{ij}=1/d_{ij}.$$
After constructing all tours: $\tau_{ij}\leftarrow(1-\rho)\tau_{ij}+\sum_a Q/L_a$ for edges used by ant $a$. Each ant contributes on **both directions** of an undirected edge.

- $\alpha$: importance of pheromone memory
- $\beta$: importance of distance heuristic
- $\rho$: evaporation fraction
- $Q$: pheromone deposition scale

### Mini exercise: compute one transition distribution
Assume an ant at city 0 has three available neighbors (1, 2, 3). Calculate their probabilities and verify they sum to one.

In [ ]:
# Step 1: Define three candidate cities and their pheromone levels

# Step 2: Set ACO parameters

# Step 3: Calculate the selection weight for each candidate city
# Higher pheromone and shorter distance produce higher weights

# Step 4: Convert weights into selection probabilities

# Step 5: Display the probability of selecting each city

# Step 6: Verify that all probabilities sum to 1



## 5. Complete ACO implementation
Read the function carefully. Identify where ants choose cities, where pheromone evaporates, and where the shortest tour is recorded.

In [ ]:
def aco_tsp(D, n_ants=20, n_iterations=100,
            alpha=1.0, beta=3.0, rho=0.5,
            Q=100.0, seed=42):

    """Ant System (AS) for symmetric TSP."""

    # Step 1: Validate ACO parameters
    
    # Step 2: Initialize random generator and number of cities

    # Step 3: Initialize pheromone levels and heuristic information

    # Heuristic information: shorter distances have higher values
 
    # Step 4: Initialize the best solution and convergence history

    # Step 5: Repeat the optimization process
    for iteration in range(n_iterations):
        tours = []
        lengths = []
        # Step 6: Each ant constructs a complete TSP route
        for ant in range(n_ants):

            # Select a random starting city
            
            # Continue until all cities have been visited
            
                # Convert weights into selection probabilities

                # Select the next city probabilistically

            # Step 7: Calculate the total distance of the ant's route

            # Update the best solution found so far
        # Step 8: Evaporate existing pheromones

        # Step 9: Deposit new pheromones on routes traveled by ants
 
        # Step 10: Record the best distance after each iteration
        history.append(best_len)

    # Return the best route, its distance, and convergence history
    return best_tour, best_len, np.asarray(history)

In [ ]:
# Step 1: Start the timer to measure computation time

# Step 2: Run ACO with a fixed random seed for reproducibility

# Step 3: Evaluate ACO performance against the known optimal distance
# A smaller optimality gap (%) indicates a better solution

# Step 4: Display the total computation time

# Step 5: Verify that the ACO route visits every city exactly once

# Step 6: Visualize the best route found by ACO

# Step 7: Plot the convergence curve
# Shows how the best distance improves over iterations


## 6. Compare with the benchmark
The published optimal tour length for `berlin52` is **7542**. ACO may or may not attain it. Interpret the gap and the cost of stochastic search.

In [ ]:
# Step 1: Create a table comparing the three solutions

# Step 2: Calculate the optimality gap (%) for each method
# A smaller gap means the route is closer to the optimal solution

# Step 3: Display the comparison table


## 7. Guided reflection (write answers)
1. What happens if $\beta=0$?
2. Why must we include the return edge?
3. What does a flat convergence curve indicate?
4. Why is a single ACO run insufficient to claim superiority?
5. How does nearest-neighbor differ from ACO in its search behavior?

**Next:** Complete the independent `eil76` assignment notebook.